In [ ]:

# Install only the packages needed by this notebook.
# Do not upgrade pandas or protobuf manually.

!pip -q install "datasets>=3.2,<5" "transformers>=4.48,<5" "accelerate>=1.2,<2" "fairlearn>=0.12,<0.14" "shap>=0.46,<0.50"


In [ ]:

# Imports, versions and reproducibility

from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
from typing import Dict, List, Tuple
import hashlib
import inspect
import json
import math
import os
import platform
import random
import shutil
import sys
import time
import warnings
import zipfile

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict, load_dataset
from fairlearn.reductions import DemographicParity, ExponentiatedGradient
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.utils.class_weight import compute_sample_weight

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

def pkg_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "not installed"

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("datasets:", pkg_version("datasets"))
print("transformers:", pkg_version("transformers"))
print("fairlearn:", pkg_version("fairlearn"))
print("scikit-learn:", pkg_version("scikit-learn"))
print("pandas:", pkg_version("pandas"))


In [ ]:

# Global configuration

DATASET_ID = "pietrolesci/civilcomments-wilds"
DATASET_CONFIG = "default"

TEXT_COLUMN = "comment_text"
TARGET_COLUMN = "toxicity"
IDENTITY_ANY_COLUMN = "identity_any"

IDENTITY_COLUMNS = [
    "male",
    "female",
    "LGBTQ",
    "christian",
    "muslim",
    "other_religions",
    "black",
    "white",
]

# Free-Colab/T4 limits
CLASSICAL_TRAIN_ROWS = 100_000
RF_TRAIN_ROWS = 50_000
COMMON_VAL_ROWS = 8_000
COMMON_TEST_ROWS = 10_000
BERT_TRAIN_ROWS = 20_000
BERT_VAL_ROWS = 4_000
MITIGATION_TRAIN_ROWS = 25_000

TFIDF_MAX_FEATURES = 25_000
BERT_MODEL_NAME = "bert-base-uncased"
BERT_MAX_LENGTH = 128
BERT_EPOCHS = 2
BERT_BATCH_SIZE = 16

MIN_FAIRNESS_GROUP_ROWS = 100

OUTPUT_ROOT = Path("/content/civilcomments_fairness_results")
TABLE_DIR = OUTPUT_ROOT / "tables"
FIGURE_DIR = OUTPUT_ROOT / "figures"
REPORT_DIR = OUTPUT_ROOT / "report_evidence"
MODEL_DIR = OUTPUT_ROOT / "temporary_models"

for directory in [TABLE_DIR, FIGURE_DIR, REPORT_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "dataset_id": DATASET_ID,
    "dataset_config": DATASET_CONFIG,
    "seed": SEED,
    "identity_columns": IDENTITY_COLUMNS,
    "classical_train_rows": CLASSICAL_TRAIN_ROWS,
    "rf_train_rows": RF_TRAIN_ROWS,
    "common_validation_rows": COMMON_VAL_ROWS,
    "common_test_rows": COMMON_TEST_ROWS,
    "bert_train_rows": BERT_TRAIN_ROWS,
    "bert_validation_rows": BERT_VAL_ROWS,
    "mitigation_train_rows": MITIGATION_TRAIN_ROWS,
    "tfidf_max_features": TFIDF_MAX_FEATURES,
    "bert_model_name": BERT_MODEL_NAME,
    "bert_max_length": BERT_MAX_LENGTH,
    "bert_epochs": BERT_EPOCHS,
    "bert_batch_size": BERT_BATCH_SIZE,
}

with open(REPORT_DIR / "experiment_config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

CONFIG


# WEEK 1 — Dataset Validation, EDA and Leakage Control

In [ ]:

# Load the verified CivilComments-WILDS processed configuration.

dataset = load_dataset(DATASET_ID, DATASET_CONFIG)

print(dataset)

EXPECTED_SPLITS = {"train", "validation", "test"}
assert EXPECTED_SPLITS.issubset(dataset.keys())

for split_name in ["train", "validation", "test"]:
    print(split_name, f"{len(dataset[split_name]):,}", dataset[split_name].column_names)


In [ ]:

# Validate research-essential schema.

CORE_COLUMNS = [
    "uid",
    "id",
    TEXT_COLUMN,
    TARGET_COLUMN,
    IDENTITY_ANY_COLUMN,
    *IDENTITY_COLUMNS,
]

schema_rows = []

for split_name in ["train", "validation", "test"]:
    columns = dataset[split_name].column_names
    missing = sorted(set(CORE_COLUMNS) - set(columns))

    schema_rows.append({
        "split": split_name,
        "rows": len(dataset[split_name]),
        "columns": len(columns),
        "core_schema_present": len(missing) == 0,
        "missing_core_columns": ", ".join(missing),
    })

schema_df = pd.DataFrame(schema_rows)
display(schema_df)

assert schema_df["core_schema_present"].all(), (
    "The selected dataset configuration does not expose the required "
    "toxicity and demographic subgroup variables."
)

print("PASS: research-essential schema is available.")


In [ ]:

# Convert official splits to pandas and prepare lightweight derived fields.

def prepare_split(split_name, hf_split):
    df = hf_split.to_pandas()

    df[TEXT_COLUMN] = df[TEXT_COLUMN].astype("string").fillna("")
    df["text_normalized"] = (
        df[TEXT_COLUMN]
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    df[TARGET_COLUMN] = pd.to_numeric(
        df[TARGET_COLUMN], errors="coerce"
    ).fillna(0).astype("int8")

    for identity in IDENTITY_COLUMNS:
        df[identity] = pd.to_numeric(
            df[identity], errors="coerce"
        ).fillna(0).astype("int8")

    df[IDENTITY_ANY_COLUMN] = pd.to_numeric(
        df[IDENTITY_ANY_COLUMN], errors="coerce"
    ).fillna(0).astype("int8")

    df["identity_count"] = df[IDENTITY_COLUMNS].sum(axis=1).astype("int8")
    df["word_count"] = (
        df["text_normalized"].str.split().str.len().fillna(0).astype("int32")
    )
    df["text_hash"] = (
        df["text_normalized"]
        .str.lower()
        .map(lambda x: hashlib.sha256(x.encode("utf-8")).hexdigest())
    )
    df["split"] = split_name
    return df

dfs = {
    split_name: prepare_split(split_name, dataset[split_name])
    for split_name in ["train", "validation", "test"]
}

for split_name, df in dfs.items():
    print(split_name, df.shape)


In [ ]:

# Dataset summary, missing values and duplicate/leakage audit.

dataset_summary_rows = []
duplicate_rows = []

for split_name, df in dfs.items():
    dataset_summary_rows.append({
        "split": split_name,
        "rows": len(df),
        "toxic_rows": int(df[TARGET_COLUMN].sum()),
        "non_toxic_rows": int((1 - df[TARGET_COLUMN]).sum()),
        "toxicity_prevalence": float(df[TARGET_COLUMN].mean()),
        "identity_reference_rows": int(df[IDENTITY_ANY_COLUMN].sum()),
        "identity_reference_prevalence": float(df[IDENTITY_ANY_COLUMN].mean()),
        "median_word_count": float(df["word_count"].median()),
    })

    duplicate_rows.append({
        "split": split_name,
        "duplicate_uid": int(df["uid"].duplicated().sum()),
        "duplicate_id": int(df["id"].duplicated().sum()),
        "duplicate_exact_text": int(df["text_hash"].duplicated().sum()),
        "empty_text": int(df["text_normalized"].str.len().eq(0).sum()),
    })

dataset_summary_df = pd.DataFrame(dataset_summary_rows)
duplicate_df = pd.DataFrame(duplicate_rows)

display(dataset_summary_df)
display(duplicate_df)

dataset_summary_df.to_csv(TABLE_DIR / "01_dataset_summary.csv", index=False)
duplicate_df.to_csv(TABLE_DIR / "02_duplicate_audit.csv", index=False)

pair_rows = []
for a, b in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    pair_rows.append({
        "split_a": a,
        "split_b": b,
        "uid_overlap": len(set(dfs[a]["uid"]) & set(dfs[b]["uid"])),
        "id_overlap": len(set(dfs[a]["id"]) & set(dfs[b]["id"])),
        "exact_text_overlap": len(set(dfs[a]["text_hash"]) & set(dfs[b]["text_hash"])),
    })

leakage_df = pd.DataFrame(pair_rows)
display(leakage_df)
leakage_df.to_csv(TABLE_DIR / "03_cross_split_leakage.csv", index=False)


In [ ]:

# Identity subgroup EDA.

subgroup_rows = []

for split_name, df in dfs.items():
    overall_rate = df[TARGET_COLUMN].mean()

    for identity in IDENTITY_COLUMNS:
        subgroup = df[df[identity] == 1]

        subgroup_rows.append({
            "split": split_name,
            "identity": identity,
            "rows": len(subgroup),
            "percentage": len(subgroup) / len(df),
            "toxic_rows": int(subgroup[TARGET_COLUMN].sum()),
            "non_toxic_rows": int((1 - subgroup[TARGET_COLUMN]).sum()),
            "toxicity_prevalence": float(subgroup[TARGET_COLUMN].mean())
                if len(subgroup) else np.nan,
            "difference_from_overall": float(
                subgroup[TARGET_COLUMN].mean() - overall_rate
            ) if len(subgroup) else np.nan,
        })

subgroup_eda_df = pd.DataFrame(subgroup_rows)
subgroup_eda_df.to_csv(TABLE_DIR / "04_identity_subgroup_eda.csv", index=False)

display(
    subgroup_eda_df[subgroup_eda_df["split"] == "train"]
    .sort_values("rows", ascending=False)
)


In [ ]:

# Week 1 report figures.

plot_df = dataset_summary_df.set_index("split")[["non_toxic_rows", "toxic_rows"]]
plot_df.columns = ["Non-toxic", "Toxic"]

ax = plot_df.plot(kind="bar", figsize=(8, 5))
ax.set_title("Toxicity class distribution by official split")
ax.set_xlabel("Split")
ax.set_ylabel("Comments")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "01_class_distribution.png", dpi=200)
plt.show()

train_subgroups = (
    subgroup_eda_df[subgroup_eda_df["split"] == "train"]
    .sort_values("toxicity_prevalence", ascending=True)
)

plt.figure(figsize=(8, 5))
plt.barh(train_subgroups["identity"], train_subgroups["toxicity_prevalence"])
plt.axvline(
    dfs["train"][TARGET_COLUMN].mean(),
    linestyle="--",
    label="Overall train prevalence",
)
plt.title("Training toxicity prevalence by identity subgroup")
plt.xlabel("Toxicity prevalence")
plt.ylabel("Identity subgroup")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / "02_subgroup_toxicity_prevalence.png", dpi=200)
plt.show()


# Shared sampling, evaluation and fairness utilities

In [ ]:

def stratified_sample(df, n_rows, seed=SEED):
    """Sample while approximately preserving toxicity and identity-any strata."""
    if n_rows is None or len(df) <= n_rows:
        return df.copy().reset_index(drop=True)

    work = df.copy()
    work["_stratum"] = (
        work[TARGET_COLUMN].astype(str)
        + "_"
        + work[IDENTITY_ANY_COLUMN].astype(str)
    )

    sampled_parts = []
    total = len(work)

    for stratum, group in work.groupby("_stratum", sort=False):
        target_n = max(1, int(round(n_rows * len(group) / total)))
        sampled_parts.append(
            group.sample(
                n=min(target_n, len(group)),
                random_state=seed,
            )
        )

    sampled = pd.concat(sampled_parts, ignore_index=True)

    if len(sampled) > n_rows:
        sampled = sampled.sample(n=n_rows, random_state=seed)

    elif len(sampled) < n_rows:
        remaining = work.drop(index=sampled.index, errors="ignore")
        need = min(n_rows - len(sampled), len(remaining))
        if need > 0:
            sampled = pd.concat(
                [sampled, remaining.sample(n=need, random_state=seed)],
                ignore_index=True,
            )

    return sampled.drop(columns=["_stratum"], errors="ignore").reset_index(drop=True)


def classification_metrics(y_true, y_pred, y_prob=None):
    result = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

    if y_prob is not None and len(np.unique(y_true)) == 2:
        result["roc_auc"] = roc_auc_score(y_true, y_prob)
    else:
        result["roc_auc"] = np.nan

    return result


def binary_rates(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    tpr = tp / (tp + fn) if (tp + fn) else np.nan
    fpr = fp / (fp + tn) if (fp + tn) else np.nan
    selection_rate = np.mean(y_pred)

    return {
        "tpr": tpr,
        "fpr": fpr,
        "selection_rate": selection_rate,
        "support": len(y_true),
    }


def fairness_by_subgroup(df, y_true, y_pred, model_name):
    """
    One-vs-rest group fairness audit.
    Differences are subgroup rate minus non-subgroup rate.
    """
    rows = []
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    for identity in IDENTITY_COLUMNS:
        group_mask = df[identity].to_numpy() == 1
        ref_mask = ~group_mask

        if group_mask.sum() < MIN_FAIRNESS_GROUP_ROWS:
            continue
        if ref_mask.sum() < MIN_FAIRNESS_GROUP_ROWS:
            continue

        group_rates = binary_rates(y_true[group_mask], y_pred[group_mask])
        ref_rates = binary_rates(y_true[ref_mask], y_pred[ref_mask])

        rows.append({
            "model": model_name,
            "identity": identity,
            "group_rows": int(group_mask.sum()),
            "reference_rows": int(ref_mask.sum()),
            "group_selection_rate": group_rates["selection_rate"],
            "reference_selection_rate": ref_rates["selection_rate"],
            "demographic_parity_difference": (
                group_rates["selection_rate"] - ref_rates["selection_rate"]
            ),
            "group_tpr": group_rates["tpr"],
            "reference_tpr": ref_rates["tpr"],
            "equal_opportunity_difference": (
                group_rates["tpr"] - ref_rates["tpr"]
            ),
            "group_fpr": group_rates["fpr"],
            "reference_fpr": ref_rates["fpr"],
            "false_positive_rate_difference": (
                group_rates["fpr"] - ref_rates["fpr"]
            ),
        })

    return pd.DataFrame(rows)


def fairness_summary(fairness_df, model_name):
    if fairness_df.empty:
        return {
            "model": model_name,
            "worst_abs_demographic_parity_difference": np.nan,
            "worst_abs_equal_opportunity_difference": np.nan,
            "worst_abs_false_positive_rate_difference": np.nan,
        }

    return {
        "model": model_name,
        "worst_abs_demographic_parity_difference": (
            fairness_df["demographic_parity_difference"].abs().max()
        ),
        "worst_abs_equal_opportunity_difference": (
            fairness_df["equal_opportunity_difference"].abs().max()
        ),
        "worst_abs_false_positive_rate_difference": (
            fairness_df["false_positive_rate_difference"].abs().max()
        ),
    }


def save_confusion_figure(y_true, y_pred, model_name, filename):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    fig, ax = plt.subplots(figsize=(5, 4))
    image = ax.imshow(cm)
    ax.set_title(f"{model_name} confusion matrix")
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_xticks([0, 1], labels=["Non-toxic", "Toxic"])
    ax.set_yticks([0, 1], labels=["Non-toxic", "Toxic"])

    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")

    fig.colorbar(image, ax=ax)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / filename, dpi=200)
    plt.show()


In [ ]:

# Fixed experiment samples.
# The common test sample is used by LR, RF and BERT for direct comparison.

classical_train_df = stratified_sample(
    dfs["train"], CLASSICAL_TRAIN_ROWS, seed=SEED
)
common_val_df = stratified_sample(
    dfs["validation"], COMMON_VAL_ROWS, seed=SEED + 1
)
common_test_df = stratified_sample(
    dfs["test"], COMMON_TEST_ROWS, seed=SEED + 2
)
bert_train_df = stratified_sample(
    dfs["train"], BERT_TRAIN_ROWS, seed=SEED + 3
)
bert_val_df = stratified_sample(
    dfs["validation"], BERT_VAL_ROWS, seed=SEED + 4
)
mitigation_train_df = stratified_sample(
    dfs["train"], MITIGATION_TRAIN_ROWS, seed=SEED + 5
)

print("Classical train:", classical_train_df.shape)
print("Common validation:", common_val_df.shape)
print("Common test:", common_test_df.shape)
print("BERT train:", bert_train_df.shape)
print("BERT validation:", bert_val_df.shape)
print("Mitigation train:", mitigation_train_df.shape)


# WEEK 2 — Classical Models and Initial Fairness Audit

In [ ]:

# TF-IDF representation shared by Logistic Regression, Random Forest and mitigation experiments.

tfidf = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.98,
    max_features=TFIDF_MAX_FEATURES,
    sublinear_tf=True,
    dtype=np.float32,
)

X_train = tfidf.fit_transform(classical_train_df["text_normalized"])
X_val = tfidf.transform(common_val_df["text_normalized"])
X_test = tfidf.transform(common_test_df["text_normalized"])

y_train = classical_train_df[TARGET_COLUMN].to_numpy()
y_val = common_val_df[TARGET_COLUMN].to_numpy()
y_test = common_test_df[TARGET_COLUMN].to_numpy()

print("TF-IDF train matrix:", X_train.shape)
print("TF-IDF validation matrix:", X_val.shape)
print("TF-IDF test matrix:", X_test.shape)


In [ ]:

# Logistic Regression baseline.

start = time.time()

lr_model = LogisticRegression(
    C=2.0,
    max_iter=1000,
    class_weight="balanced",
    solver="liblinear",
    random_state=SEED,
)

lr_model.fit(X_train, y_train)

lr_training_seconds = time.time() - start

lr_prob = lr_model.predict_proba(X_test)[:, 1]
lr_pred = (lr_prob >= 0.50).astype(int)

lr_metrics = classification_metrics(y_test, lr_pred, lr_prob)
lr_metrics.update({
    "model": "Logistic Regression",
    "training_seconds": lr_training_seconds,
    "test_rows": len(y_test),
})

print(lr_metrics)
save_confusion_figure(
    y_test,
    lr_pred,
    "Logistic Regression",
    "03_lr_confusion_matrix.png",
)

lr_fairness = fairness_by_subgroup(
    common_test_df,
    y_test,
    lr_pred,
    "Logistic Regression",
)

display(lr_fairness)


In [ ]:

# Random Forest baseline.
# Uses a smaller training subset for free-Colab feasibility.

rf_train_df = stratified_sample(
    classical_train_df,
    RF_TRAIN_ROWS,
    seed=SEED + 6,
)

rf_indices = rf_train_df.index.to_numpy()
X_rf_train = tfidf.transform(rf_train_df["text_normalized"])
y_rf_train = rf_train_df[TARGET_COLUMN].to_numpy()

start = time.time()

rf_model = RandomForestClassifier(
    n_estimators=180,
    max_depth=35,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=SEED,
)

rf_model.fit(X_rf_train, y_rf_train)

rf_training_seconds = time.time() - start

rf_prob = rf_model.predict_proba(X_test)[:, 1]
rf_pred = (rf_prob >= 0.50).astype(int)

rf_metrics = classification_metrics(y_test, rf_pred, rf_prob)
rf_metrics.update({
    "model": "Random Forest",
    "training_seconds": rf_training_seconds,
    "test_rows": len(y_test),
})

print(rf_metrics)
save_confusion_figure(
    y_test,
    rf_pred,
    "Random Forest",
    "04_rf_confusion_matrix.png",
)

rf_fairness = fairness_by_subgroup(
    common_test_df,
    y_test,
    rf_pred,
    "Random Forest",
)

display(rf_fairness)


In [ ]:

# Week 2 baseline result tables.

baseline_performance_df = pd.DataFrame([
    lr_metrics,
    rf_metrics,
])

baseline_fairness_df = pd.concat(
    [lr_fairness, rf_fairness],
    ignore_index=True,
)

baseline_fairness_summary_df = pd.DataFrame([
    fairness_summary(lr_fairness, "Logistic Regression"),
    fairness_summary(rf_fairness, "Random Forest"),
])

display(baseline_performance_df)
display(baseline_fairness_summary_df)

baseline_performance_df.to_csv(
    TABLE_DIR / "05_baseline_model_performance.csv",
    index=False,
)
baseline_fairness_df.to_csv(
    TABLE_DIR / "06_baseline_subgroup_fairness.csv",
    index=False,
)


# WEEK 3 — BERT and Bias-Mitigation Experiments


## 3A. Fine-tuned BERT

The proposal specifies BERT, so this notebook uses `bert-base-uncased`.
The reduced sample size, sequence length 128, mixed precision and two epochs are selected
to fit a free T4.

If CUDA is unavailable, stop here and enable a T4 before training BERT.


In [ ]:

assert torch.cuda.is_available(), (
    "BERT training requires a Colab GPU for this project. "
    "Select Runtime > Change runtime type > T4 GPU."
)

tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)

def to_hf_dataset(df):
    return Dataset.from_pandas(
        pd.DataFrame({
            "text": df["text_normalized"].astype(str),
            "label": df[TARGET_COLUMN].astype(int),
        }),
        preserve_index=False,
    )

bert_train_ds = to_hf_dataset(bert_train_df)
bert_val_ds = to_hf_dataset(bert_val_df)
bert_test_ds = to_hf_dataset(common_test_df)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=BERT_MAX_LENGTH,
    )

bert_train_ds = bert_train_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"],
)
bert_val_ds = bert_val_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"],
)
bert_test_ds = bert_test_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"],
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8,
)


In [ ]:

# Trainer compatibility helper for Transformers 4.x versions.

def bert_compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(
        torch.tensor(logits),
        dim=1,
    ).numpy()[:, 1]
    preds = (probs >= 0.50).astype(int)
    return classification_metrics(labels, preds, probs)


training_args_kwargs = dict(
    output_dir=str(MODEL_DIR / "bert_checkpoints"),
    num_train_epochs=BERT_EPOCHS,
    per_device_train_batch_size=BERT_BATCH_SIZE,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.10,
    fp16=True,
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=1,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

training_arg_params = inspect.signature(
    TrainingArguments.__init__
).parameters

if "eval_strategy" in training_arg_params:
    training_args_kwargs["eval_strategy"] = "epoch"
else:
    training_args_kwargs["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(
    **training_args_kwargs
)

bert_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_MODEL_NAME,
    num_labels=2,
)

trainer_kwargs = dict(
    model=bert_model,
    args=training_args,
    train_dataset=bert_train_ds,
    eval_dataset=bert_val_ds,
    data_collator=data_collator,
    compute_metrics=bert_compute_metrics,
)

trainer_params = inspect.signature(
    Trainer.__init__
).parameters

if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_params:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)


In [ ]:

# Train BERT.

start = time.time()
train_output = trainer.train()
bert_training_seconds = time.time() - start

print("BERT training seconds:", bert_training_seconds)
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation metric:", trainer.state.best_metric)

bert_log_df = pd.DataFrame(trainer.state.log_history)
bert_log_df.to_csv(
    TABLE_DIR / "07_bert_training_log.csv",
    index=False,
)


In [ ]:

# Evaluate BERT on the same common test sample used for LR and RF.

bert_prediction_output = trainer.predict(bert_test_ds)

bert_logits = bert_prediction_output.predictions
bert_prob = torch.softmax(
    torch.tensor(bert_logits),
    dim=1,
).numpy()[:, 1]

bert_pred = (bert_prob >= 0.50).astype(int)

bert_metrics = classification_metrics(
    y_test,
    bert_pred,
    bert_prob,
)

bert_metrics.update({
    "model": "BERT",
    "training_seconds": bert_training_seconds,
    "test_rows": len(y_test),
})

print(bert_metrics)

save_confusion_figure(
    y_test,
    bert_pred,
    "BERT",
    "05_bert_confusion_matrix.png",
)

bert_fairness = fairness_by_subgroup(
    common_test_df,
    y_test,
    bert_pred,
    "BERT",
)

display(bert_fairness)



## 3B. Controlled fairness mitigation comparison

To isolate the effect of mitigation, all three mitigation approaches use the same
TF-IDF Logistic Regression family.

The sensitive feature used during mitigation is `identity_any`:
- `0`: none of the eight WILDS identity indicators are active;
- `1`: at least one identity indicator is active.

After mitigation, fairness is still audited separately across **all eight demographic
subgroups**. This shows whether improving the aggregate identity-reference disparity also
helps or harms specific identities.


In [ ]:

# Mitigation matrices use the already-fitted TF-IDF representation.

X_mit_train = tfidf.transform(mitigation_train_df["text_normalized"])
y_mit_train = mitigation_train_df[TARGET_COLUMN].to_numpy()
a_mit_train = mitigation_train_df[IDENTITY_ANY_COLUMN].to_numpy()

X_mit_val = X_val
y_mit_val = y_val
a_mit_val = common_val_df[IDENTITY_ANY_COLUMN].to_numpy()

X_mit_test = X_test
y_mit_test = y_test
a_mit_test = common_test_df[IDENTITY_ANY_COLUMN].to_numpy()


In [ ]:

# Pre-processing mitigation: reweighing by sensitive group x target.
# Weight = P(A=a)P(Y=y) / P(A=a,Y=y)

def reweighing_weights(sensitive, target):
    sensitive = np.asarray(sensitive)
    target = np.asarray(target)
    n = len(target)

    weights = np.ones(n, dtype=float)

    for a in np.unique(sensitive):
        for y in np.unique(target):
            mask_a = sensitive == a
            mask_y = target == y
            mask_ay = mask_a & mask_y

            p_a = mask_a.mean()
            p_y = mask_y.mean()
            p_ay = mask_ay.mean()

            if p_ay > 0:
                weights[mask_ay] = (p_a * p_y) / p_ay

    return weights


pre_weights = reweighing_weights(
    a_mit_train,
    y_mit_train,
)

pre_model = LogisticRegression(
    C=2.0,
    max_iter=1000,
    solver="liblinear",
    random_state=SEED,
)

pre_model.fit(
    X_mit_train,
    y_mit_train,
    sample_weight=pre_weights,
)

pre_prob = pre_model.predict_proba(X_mit_test)[:, 1]
pre_pred = (pre_prob >= 0.50).astype(int)

pre_metrics = classification_metrics(
    y_mit_test,
    pre_pred,
    pre_prob,
)
pre_metrics["model"] = "LR + Pre-processing Reweighing"

pre_fairness = fairness_by_subgroup(
    common_test_df,
    y_mit_test,
    pre_pred,
    "LR + Pre-processing Reweighing",
)

print(pre_metrics)


In [ ]:

# In-processing mitigation: Fairlearn ExponentiatedGradient with DemographicParity.

base_estimator = LogisticRegression(
    C=1.0,
    max_iter=500,
    solver="liblinear",
    random_state=SEED,
)

constraint = DemographicParity(
    difference_bound=0.05
)

in_model = ExponentiatedGradient(
    estimator=base_estimator,
    constraints=constraint,
    eps=0.05,
    max_iter=20,
)

start = time.time()

in_model.fit(
    X_mit_train,
    y_mit_train,
    sensitive_features=a_mit_train,
)

in_training_seconds = time.time() - start

# ExponentiatedGradient exposes randomized predictions.
# Fix the NumPy seed immediately before prediction for reproducibility.
np.random.seed(SEED)
in_pred = np.asarray(
    in_model.predict(X_mit_test)
).astype(int)

in_metrics = classification_metrics(
    y_mit_test,
    in_pred,
    y_prob=None,
)
in_metrics["model"] = "LR + In-processing ExponentiatedGradient"
in_metrics["training_seconds"] = in_training_seconds

in_fairness = fairness_by_subgroup(
    common_test_df,
    y_mit_test,
    in_pred,
    "LR + In-processing ExponentiatedGradient",
)

print(in_metrics)


In [ ]:

# Post-processing mitigation:
# validation-selected group-specific thresholds for identity_any.
# The test set is never used for threshold selection.

post_base_model = LogisticRegression(
    C=2.0,
    max_iter=1000,
    class_weight="balanced",
    solver="liblinear",
    random_state=SEED,
)

post_base_model.fit(
    X_mit_train,
    y_mit_train,
)

val_prob = post_base_model.predict_proba(X_mit_val)[:, 1]
test_prob = post_base_model.predict_proba(X_mit_test)[:, 1]

def apply_group_thresholds(probabilities, sensitive, threshold_0, threshold_1):
    probabilities = np.asarray(probabilities)
    sensitive = np.asarray(sensitive)

    thresholds = np.where(
        sensitive == 1,
        threshold_1,
        threshold_0,
    )
    return (probabilities >= thresholds).astype(int)


def aggregate_identity_any_gap(y_true, y_pred, sensitive):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    sensitive = np.asarray(sensitive)

    rates_0 = binary_rates(
        y_true[sensitive == 0],
        y_pred[sensitive == 0],
    )
    rates_1 = binary_rates(
        y_true[sensitive == 1],
        y_pred[sensitive == 1],
    )

    return (
        abs(rates_1["tpr"] - rates_0["tpr"])
        + abs(rates_1["fpr"] - rates_0["fpr"])
    )


threshold_grid = np.round(
    np.arange(0.25, 0.76, 0.05),
    2,
)

best_post = None

for threshold_0 in threshold_grid:
    for threshold_1 in threshold_grid:
        pred = apply_group_thresholds(
            val_prob,
            a_mit_val,
            threshold_0,
            threshold_1,
        )

        fairness_gap = aggregate_identity_any_gap(
            y_mit_val,
            pred,
            a_mit_val,
        )

        model_f1 = f1_score(
            y_mit_val,
            pred,
            zero_division=0,
        )

        # Primary goal: reduce equalized-odds style gap.
        # Secondary penalty prevents collapsing predictive usefulness.
        objective = fairness_gap + 0.35 * (1 - model_f1)

        candidate = {
            "threshold_0": threshold_0,
            "threshold_1": threshold_1,
            "fairness_gap": fairness_gap,
            "validation_f1": model_f1,
            "objective": objective,
        }

        if best_post is None or candidate["objective"] < best_post["objective"]:
            best_post = candidate

print("Selected validation thresholds:", best_post)

post_pred = apply_group_thresholds(
    test_prob,
    a_mit_test,
    best_post["threshold_0"],
    best_post["threshold_1"],
)

post_metrics = classification_metrics(
    y_mit_test,
    post_pred,
    test_prob,
)
post_metrics["model"] = "LR + Post-processing Thresholds"

post_fairness = fairness_by_subgroup(
    common_test_df,
    y_mit_test,
    post_pred,
    "LR + Post-processing Thresholds",
)

print(post_metrics)


In [ ]:

# Week 3 mitigation comparison.

mitigation_performance_df = pd.DataFrame([
    {
        **classification_metrics(y_mit_test, lr_pred, lr_prob),
        "model": "Unmitigated Logistic Regression",
    },
    pre_metrics,
    in_metrics,
    post_metrics,
])

mitigation_fairness_detail_df = pd.concat(
    [
        lr_fairness.assign(model="Unmitigated Logistic Regression"),
        pre_fairness,
        in_fairness,
        post_fairness,
    ],
    ignore_index=True,
)

mitigation_fairness_summary_df = pd.DataFrame([
    fairness_summary(
        lr_fairness,
        "Unmitigated Logistic Regression",
    ),
    fairness_summary(
        pre_fairness,
        "LR + Pre-processing Reweighing",
    ),
    fairness_summary(
        in_fairness,
        "LR + In-processing ExponentiatedGradient",
    ),
    fairness_summary(
        post_fairness,
        "LR + Post-processing Thresholds",
    ),
])

mitigation_comparison_df = mitigation_performance_df.merge(
    mitigation_fairness_summary_df,
    on="model",
    how="left",
)

display(mitigation_comparison_df)

mitigation_comparison_df.to_csv(
    TABLE_DIR / "08_mitigation_comparison.csv",
    index=False,
)
mitigation_fairness_detail_df.to_csv(
    TABLE_DIR / "09_mitigation_subgroup_fairness.csv",
    index=False,
)


# WEEK 4 — Final Comparison, Explainability and Report Evidence

In [ ]:

# Combine LR, RF and BERT performance and fairness.

all_model_performance_df = pd.DataFrame([
    lr_metrics,
    rf_metrics,
    bert_metrics,
])

all_model_fairness_detail_df = pd.concat(
    [
        lr_fairness,
        rf_fairness,
        bert_fairness,
    ],
    ignore_index=True,
)

all_model_fairness_summary_df = pd.DataFrame([
    fairness_summary(
        lr_fairness,
        "Logistic Regression",
    ),
    fairness_summary(
        rf_fairness,
        "Random Forest",
    ),
    fairness_summary(
        bert_fairness,
        "BERT",
    ),
])

final_model_comparison_df = all_model_performance_df.merge(
    all_model_fairness_summary_df,
    on="model",
    how="left",
)

display(final_model_comparison_df)

all_model_performance_df.to_csv(
    TABLE_DIR / "10_final_model_performance.csv",
    index=False,
)
all_model_fairness_detail_df.to_csv(
    TABLE_DIR / "11_final_subgroup_fairness.csv",
    index=False,
)
final_model_comparison_df.to_csv(
    TABLE_DIR / "12_final_model_comparison.csv",
    index=False,
)


In [ ]:

# Final performance chart.

plot_df = final_model_comparison_df.set_index("model")[
    ["accuracy", "precision", "recall", "f1", "roc_auc"]
]

ax = plot_df.plot(
    kind="bar",
    figsize=(10, 6),
)
ax.set_ylim(0, 1)
ax.set_title("Model classification performance")
ax.set_ylabel("Score")
ax.set_xlabel("Model")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "06_model_performance_comparison.png",
    dpi=200,
)
plt.show()


In [ ]:

# Final fairness disparity chart.

fair_plot = final_model_comparison_df.set_index("model")[
    [
        "worst_abs_demographic_parity_difference",
        "worst_abs_equal_opportunity_difference",
        "worst_abs_false_positive_rate_difference",
    ]
]

fair_plot.columns = [
    "Demographic parity",
    "Equal opportunity",
    "False-positive rate",
]

ax = fair_plot.plot(
    kind="bar",
    figsize=(10, 6),
)
ax.set_title("Worst absolute demographic disparity by model")
ax.set_ylabel("Absolute disparity")
ax.set_xlabel("Model")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "07_model_fairness_comparison.png",
    dpi=200,
)
plt.show()


In [ ]:

# Fairness-performance trade-off for mitigation methods.

tradeoff_df = mitigation_comparison_df.copy()
tradeoff_df["combined_fairness_disparity"] = tradeoff_df[
    [
        "worst_abs_equal_opportunity_difference",
        "worst_abs_false_positive_rate_difference",
    ]
].mean(axis=1)

plt.figure(figsize=(8, 6))

for _, row in tradeoff_df.iterrows():
    plt.scatter(
        row["combined_fairness_disparity"],
        row["f1"],
        s=80,
    )
    plt.annotate(
        row["model"].replace("Logistic Regression", "LR"),
        (
            row["combined_fairness_disparity"],
            row["f1"],
        ),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8,
    )

plt.xlabel("Mean absolute EO/FPR disparity (lower is fairer)")
plt.ylabel("F1 score (higher is better)")
plt.title("Fairness–performance trade-off")
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "08_fairness_performance_tradeoff.png",
    dpi=200,
)
plt.show()


## Explainability

In [ ]:

# Logistic Regression global feature importance.
# This is always produced and is inexpensive.

feature_names = np.asarray(
    tfidf.get_feature_names_out()
)
coefficients = lr_model.coef_[0]

top_positive_indices = np.argsort(
    coefficients
)[-30:][::-1]

top_negative_indices = np.argsort(
    coefficients
)[:30]

lr_feature_importance_df = pd.concat(
    [
        pd.DataFrame({
            "feature": feature_names[top_positive_indices],
            "coefficient": coefficients[top_positive_indices],
            "direction": "towards_toxic",
        }),
        pd.DataFrame({
            "feature": feature_names[top_negative_indices],
            "coefficient": coefficients[top_negative_indices],
            "direction": "towards_non_toxic",
        }),
    ],
    ignore_index=True,
)

display(lr_feature_importance_df)

lr_feature_importance_df.to_csv(
    TABLE_DIR / "13_lr_top_features.csv",
    index=False,
)


In [ ]:

# Lightweight SHAP attempt for the Logistic Regression model.
# Failure here does not invalidate the project; status is explicitly recorded.

shap_status = {
    "status": "not_started",
    "details": "",
}

try:
    import shap

    # Explain a small sparse test sample to keep Colab memory use low.
    shap_sample_rows = min(100, X_test.shape[0])
    X_shap = X_test[:shap_sample_rows]

    explainer = shap.LinearExplainer(
        lr_model,
        X_train[: min(500, X_train.shape[0])],
    )
    shap_values = explainer.shap_values(X_shap)

    if isinstance(shap_values, list):
        shap_array = np.asarray(shap_values[-1])
    else:
        shap_array = np.asarray(shap_values)

    mean_abs_shap = np.asarray(
        np.abs(shap_array).mean(axis=0)
    ).ravel()

    top_shap_indices = np.argsort(
        mean_abs_shap
    )[-30:][::-1]

    shap_summary_df = pd.DataFrame({
        "feature": feature_names[top_shap_indices],
        "mean_abs_shap": mean_abs_shap[top_shap_indices],
    })

    shap_summary_df.to_csv(
        TABLE_DIR / "14_lr_shap_top_features.csv",
        index=False,
    )

    shap_status = {
        "status": "completed",
        "details": f"Explained {shap_sample_rows} test records.",
    }

except Exception as error:
    shap_status = {
        "status": "fallback_to_coefficients",
        "details": (
            f"{type(error).__name__}: {error}. "
            "Use 13_lr_top_features.csv as the explainability evidence."
        ),
    }

print(shap_status)

with open(
    REPORT_DIR / "shap_status.json",
    "w",
) as f:
    json.dump(shap_status, f, indent=2)


## Error analysis

In [ ]:

def build_error_analysis(
    df,
    y_true,
    y_pred,
    y_prob,
    model_name,
    max_each=20,
):
    working = pd.DataFrame({
        "record_id": df["uid"].astype(str).to_numpy(),
        "text": df["text_normalized"].astype(str).to_numpy(),
        "true_label": np.asarray(y_true),
        "predicted_label": np.asarray(y_pred),
        "toxic_probability": np.asarray(y_prob),
    })

    working["model"] = model_name
    working["error_type"] = "correct"

    working.loc[
        (working["true_label"] == 0)
        & (working["predicted_label"] == 1),
        "error_type",
    ] = "false_positive"

    working.loc[
        (working["true_label"] == 1)
        & (working["predicted_label"] == 0),
        "error_type",
    ] = "false_negative"

    false_positive = (
        working[working["error_type"] == "false_positive"]
        .sort_values("toxic_probability", ascending=False)
        .head(max_each)
    )

    false_negative = (
        working[working["error_type"] == "false_negative"]
        .sort_values("toxic_probability", ascending=True)
        .head(max_each)
    )

    return pd.concat(
        [false_positive, false_negative],
        ignore_index=True,
    )


lr_errors = build_error_analysis(
    common_test_df,
    y_test,
    lr_pred,
    lr_prob,
    "Logistic Regression",
)

bert_errors = build_error_analysis(
    common_test_df,
    y_test,
    bert_pred,
    bert_prob,
    "BERT",
)

error_analysis_df = pd.concat(
    [lr_errors, bert_errors],
    ignore_index=True,
)

display(
    error_analysis_df[
        ["model", "error_type", "true_label", "predicted_label", "toxic_probability"]
    ].head(20)
)

error_analysis_df.to_csv(
    TABLE_DIR / "15_error_analysis.csv",
    index=False,
)


## Automatic final result summary

In [ ]:

# Generate a concise machine-readable summary

best_f1_row = final_model_comparison_df.loc[
    final_model_comparison_df["f1"].idxmax()
]

fairest_eo_row = final_model_comparison_df.loc[
    final_model_comparison_df[
        "worst_abs_equal_opportunity_difference"
    ].idxmin()
]

best_mitigation_row = mitigation_comparison_df.loc[
    mitigation_comparison_df[
        "worst_abs_equal_opportunity_difference"
    ].idxmin()
]

summary_text = f"""
# Final Experimental Summary

## Dataset
CivilComments-WILDS processed default configuration.

## Classification comparison
Best F1 model: {best_f1_row['model']}
F1: {best_f1_row['f1']:.4f}
Accuracy: {best_f1_row['accuracy']:.4f}
Recall: {best_f1_row['recall']:.4f}
ROC-AUC: {best_f1_row['roc_auc']:.4f}

## Baseline fairness comparison
Lowest worst absolute equal-opportunity disparity:
{fairest_eo_row['model']}
Worst |Equal Opportunity Difference|:
{fairest_eo_row['worst_abs_equal_opportunity_difference']:.4f}

## Mitigation comparison
Lowest worst absolute equal-opportunity disparity:
{best_mitigation_row['model']}
F1: {best_mitigation_row['f1']:.4f}
Worst |Equal Opportunity Difference|:
{best_mitigation_row['worst_abs_equal_opportunity_difference']:.4f}
Worst |False Positive Rate Difference|:
{best_mitigation_row['worst_abs_false_positive_rate_difference']:.4f}

## SHAP
Status: {shap_status['status']}
Details: {shap_status['details']}

## Interpretation rule
The final dissertation should not select a model solely by accuracy.
The research question requires comparison of predictive performance
against demographic disparity and the fairness cost/benefit of mitigation.
""".strip()

summary_path = REPORT_DIR / "final_experimental_summary.md"
summary_path.write_text(summary_text)

print(summary_text)
